In [8]:
import pandas as pd
import numpy as np 

data = pd.read_csv("./data/clean_weather.csv", index_col=0)
data = data.ffill()

data.tail(3)

,tmax,tmin,rain,tmax_tomorrow
2022-11-24,66.0,41.0,0.0,70.0
2022-11-25,70.0,39.0,0.0,62.0
2022-11-26,62.0,41.0,0.0,64.0


Above, we initialized our network parameters. We used the numpy random.rand function to randomly create parameter matrices of a certain shape. This network will take in a single input feature, turn it into 2 hidden features, and output one prediction.

In [9]:
def mse_grad(y, y_pred):
    return (y_pred - y)

In [13]:
from scipy.special import softmax 
from sklearn.metrics import mean_squared_error as mse

class RNN():
    def __init__(self, hidden_size, time_steps, epochs=1000):
        self.hidden_size = hidden_size
        self.time_steps = time_steps
        self.epochs = epochs
        # Get TIME_STEPS temperature values from our data
        self.x = data['tmax'].tail(time_steps).to_numpy()
        self.y = data['tmax_tomorrow'].tail(time_steps).to_numpy()
    
    
    def InitWeights(self):
        np.random.seed(0) # Set a random seed so the numbers are the same
        n = self.hidden_size
        
        # Define our weights and biases

        # .rand(rows, columns)
        # Scale them down so values get through the tanh nonlinearity
        self.U = np.random.rand(1, n) / n - .1 # Input weights connection - (1 x n)
        self.W = np.random.rand(n, n) / n - .1 # Hidden to hidden weight connections - (n x n)
        self.b = np.random.rand(1, n) / n - .1 # 

        # Tanh pushes values to between -1 and 1, so scale up the output weights
        self.V = np.random.rand(n, 1) * (n * 10) # Output weight connections - (n x 1)
        self.c = np.random.rand(1, 1) # 
    
    def ForwardPass(self, xt, prevH):
        at = self.b + (xt @ self.U) + (prevH @ self.W)
        ht = np.tanh(at) 
        ot = self.c + (ht @ self.V)
        return (ht, ot)
    
    def FeedForward(self):  
        ts = self.time_steps
        hs = self.hidden_size
        
        # store the previous hidden state, since we'll need it to calculate the current hidden step
        prev_hidden =  np.zeros((1, hs))

        # An array to store the output predictions
        self.outputs = np.zeros(ts)
        

        # An array to store hidden states for use in backpropagation
        self.hiddens = np.zeros((ts, hs))

        for i in range(ts):
            xt = self.x[i].reshape(1, 1)
            ht, ot = self.ForwardPass(xt, prev_hidden)
            
            # Updates hidden states
            prev_hidden = ht
            self.hiddens[i,] = ht
            
            # Stores the output prediction
            self.outputs[i] = ot.item()
        
        self.loss = mse(self.y, self.outputs)
                           

    def BackPropagation(self):
        ts = self.time_steps
        V_grad, c_grad, W_grad, b_grad, U_grad = [0] * 5
        next_hidden = None 
        
        loss_gradients = mse_grad(self.y, self.outputs)
        for t in range((ts - 1), -1, -1):
            # gradient for the output layer
            o_grad = loss_gradients[t].reshape(1, 1)            # ∇o(t) 
            
            # output weights and bias
            V_grad += self.hiddens[t][:, np.newaxis] @ o_grad   # ∇V = Σ (∇o(t) . T[h(t)])
            c_grad += np.mean(o_grad)                           # ∇c = Σ (∇o(t))
            
            # reversing the multiplication so we can put our gradient down to the hidden step
            ho_grad = o_grad @ self.V.T                         # ∇HO = ∇o(t) . T[V]
            

            #  add together the gradients coming from output and from next hidden step
            if next_hidden is not None:
                hh_grad = next_hidden @ self.W.T                # ∇HH = ∇h(t + 1) . T[W]
                h_grad = ho_grad + hh_grad                      # ∇h(t) = ∇HH + ∇HO
            else:
                h_grad = ho_grad
                
            # undo tanh so we can get gradient values inside the hyperbolic tangent    
            tanh_deriv = 1 - self.hiddens[t, :][np.newaxis, :] ** 2 # d(tanh(at))/dt) = 1 - h(t)**2 
        
            H_grad = np.multiply(h_grad, tanh_deriv)                # ∇H = d(tanh(at))/dt) . ∇h(t)
            
            next_hidden = H_grad

            # compute our gradient wrt our hidden weights
            if t > 0:
                W_grad += self.hiddens[(t - 1), :][:, np.newaxis] @ H_grad  # ∇W = Σ ∇H . T[h(t-1)]
                b_grad += np.mean(H_grad)                                   # ∇b = Σ ∇H 
                
                U_grad += self.x[t].reshape(1, 1).T @ H_grad               # ∇U = Σ ∇H . T[x(t)]
        return (V_grad, c_grad, W_grad, b_grad, U_grad)
    
    def UpdateParameters(self):
        V_grad, c_grad, W_grad, b_grad, U_grad = self.BackPropagation()         
        lr = 1e-6
        
        # We'll divide the learning rate by the sequence length, since we were adding together the gradients
        # This makes training the model more stable
        lr /= self.time_steps
    
        self.V -= lr * V_grad
        self.c -= lr * c_grad
        self.W -= lr * W_grad
        self.b -= lr * b_grad
        self.U -= lr * U_grad
        
    def TrainModel(self):
        self.InitWeights()

        for i in range(self.epochs):
            self.FeedForward()
            self.BackPropagation()
            self.UpdateParameters()
            print(f" ({i}/{self.epochs}) | MSE = {self.loss}")


In [14]:
rnn = RNN(hidden_size = 5,
          time_steps = 3,
          epochs = 100)
rnn.TrainModel()

 (0/100) | MSE = 178.98762845849993
 (1/100) | MSE = 6711.39520599634
 (2/100) | MSE = 3727.65231124519
 (3/100) | MSE = 1632.899059835175
 (4/100) | MSE = 1631.636627600508
 (5/100) | MSE = 1630.1457367249677
 (6/100) | MSE = 1628.358182376697
 (7/100) | MSE = 1626.1761873446956
 (8/100) | MSE = 1623.454756345606
 (9/100) | MSE = 1619.9699475652362
 (10/100) | MSE = 1615.3582316575842
 (11/100) | MSE = 1608.9915566700117
 (12/100) | MSE = 1599.694425538384
 (13/100) | MSE = 1585.0195321892297
 (14/100) | MSE = 1559.0592445782895
 (15/100) | MSE = 1504.0882824684368
 (16/100) | MSE = 1345.5539865877713
 (17/100) | MSE = 622.6921575117794
 (18/100) | MSE = 287.6658787005264
 (19/100) | MSE = 189.8502152742836
 (20/100) | MSE = 48.78094390629144
 (21/100) | MSE = 12.604199794068336
 (22/100) | MSE = 36.18597883102731
 (23/100) | MSE = 12.99712380188175
 (24/100) | MSE = 31.281733290389525
 (25/100) | MSE = 13.50755301781642
 (26/100) | MSE = 28.40573022559521
 (27/100) | MSE = 14.0142647